# GPT Decoder-Only Transformer — Next-Token Prediction

**Goal:** Build and train a small GPT-style Decoder-only Transformer from scratch in PyTorch, then test next-token generation and save the model.

**Important:** This is an educational small model trained on a tiny sample text corpus. It demonstrates the architecture; it is not ChatGPT and will not produce broad, reliable knowledge.

## Notebook roadmap
1. Check the Colab runtime and install/import libraries
2. Prepare a small text dataset
3. Build a character-level tokenizer
4. Create training examples for next-token prediction
5. Implement causal self-attention and the Decoder-only Transformer
6. Train and evaluate the model
7. Generate text and predict the next token
8. Save model files for later deployment

After training, we will prepare a separate API and Vercel frontend. Vercel will host the website; the PyTorch model should run on a model-hosting service rather than inside a standard Vercel frontend deployment.

## 1. Check runtime

### Explanation
A runtime is the computer that executes notebook cells. Colab can provide a GPU, which can make neural-network training faster. `torch.cuda.is_available()` checks whether PyTorch can use that GPU. If it prints CPU, the notebook can still run, but training may take longer.

In [ ]:
import torch, sys, os, json, math, random
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("GPU not detected. Using CPU.")
print("Device:", device)

## 2. Imports and reproducibility

### Explanation
- `torch` is the PyTorch library used to create and train the neural network.
- `nn` contains ready-made neural-network layers.
- `F` contains functions such as softmax and cross-entropy loss.
- A random seed makes the example more repeatable.
- `OUT_DIR` is the folder where we will save the trained model and tokenizer.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
from pathlib import Path

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_DIR = Path("/content/gpt_project")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT_DIR)

## 3. Prepare the training text

This starter corpus is included so the notebook runs immediately. Later, you can replace `text_data` with your own permitted text dataset. Keep the training text relevant to the kind of text you want the model to learn.

### Explanation
A language model learns patterns from examples. The starter dataset is a small collection of sentences about AI and Transformers. It is included so the notebook can run without downloading a dataset. A small dataset is useful for learning the code, but it does not teach the model broad language knowledge. Later, you can replace it with a larger, suitable text dataset that you have permission to use.

In [ ]:
text_data = """
Artificial intelligence is a field of computer science.
A language model learns patterns from examples of text.
A transformer uses attention to process information.
A decoder predicts the next token using previous tokens.
A token can be a character, a word, or part of a word.
The model reads a sequence and learns to continue the sequence.
Training adjusts model parameters to reduce prediction error.
Attention helps each position use information from earlier positions.
A causal mask prevents a token from looking at future tokens.
The query, key, and value vectors are used in self attention.
The query represents what a token is looking for.
The key represents information that a token offers.
The value contains information that can be passed forward.
A feed forward network processes each token representation.
An embedding converts token IDs into numerical vectors.
Positional information tells the model where tokens appear.
The final layer produces scores for possible next tokens.
Softmax converts scores into a probability distribution.
The model predicts a likely next character from the context.
""".strip()

print("Characters in dataset:", len(text_data))
print(text_data[:500])

## 4. Build a simple character-level tokenizer

For this beginner project, each character is one token. This keeps the model small and makes next-token prediction easy to inspect.

### Explanation
A tokenizer converts text into numbers because a neural network works with numeric inputs. This notebook uses a **character-level tokenizer**:
- `chars` stores the unique characters in the dataset.
- `stoi` maps each character to an integer ID.
- `itos` maps an integer ID back to its character.
- `encode()` converts text into IDs.
- `decode()` converts IDs back into text.

For example, the word `cat` becomes three character tokens. This is simpler than the subword tokenizers used by modern GPT systems.

In [ ]:
chars = sorted(list(set(text_data)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s if ch in stoi]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text_data), dtype=torch.long)
print("Vocabulary size:", vocab_size)
print("First 30 tokens:", encode(text_data[:30]))
print("Decoded check:", decode(encode("Artificial intelligence")))

## 5. Split data and create batches

The model learns: given a context of previous characters, predict the next character.

### Explanation
We split the text into training data and validation data. The model learns from the training part; validation data gives us a separate check of how well it predicts text it did not train directly on.

`block_size` is the maximum context length. `batch_size` is how many training sequences are processed together. For every input sequence `x`, target sequence `y` is shifted one character to the left. Therefore, at each position, the correct answer is the **next character**.

In [ ]:
split_at = int(0.90 * len(data))
train_data = data[:split_at]
val_data = data[split_at:]

block_size = 64
batch_size = 32

def get_batch(split):
    source = train_data if split == "train" else val_data
    # Ensure there are enough characters for a full context and target.
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)
print("Example input:", repr(decode(xb[0].tolist())))
print("Example target:", repr(decode(yb[0].tolist())))

## 6. Define the Decoder-only Transformer

The model contains:
- Token embeddings and positional embeddings
- Masked (causal) multi-head self-attention
- Feed-forward network
- Residual connections and layer normalization
- Linear output layer that predicts the next token

The causal mask ensures a position cannot use future tokens.

### Explanation of the architecture

The main flow is:

**Token IDs → Token embeddings + Position embeddings → Decoder blocks → LayerNorm → Linear output scores**

Inside each decoder block:
1. **Masked self-attention:** lets each position use information from itself and earlier positions, but not future positions.
2. **Feed-forward network:** processes each position's representation.
3. **Residual connections:** add the block input back to the sublayer output, helping information flow through the network.
4. **Layer normalization:** helps keep activations well-scaled during training.

**Q, K and V in simple words**
- **Query (Q):** what this token is looking for.
- **Key (K):** what information this token offers for matching.
- **Value (V):** the information that is passed along when attention is calculated.

Attention compares queries with keys to calculate weights, then uses those weights to combine values. Multiple attention heads learn different relationships.

**Why is it Decoder-only?** The model uses causal (masked) self-attention and predicts the next token. It does not have a separate encoder.

**Important:** This is a compact educational GPT-style decoder. It demonstrates the central ideas, not every engineering detail of a production GPT model.

In [ ]:
# Model settings: intentionally small so it can train in a free Colab runtime.
batch_size = 32
block_size = 64
max_iters = 1500
eval_interval = 300
learning_rate = 3e-4
n_embd = 96
n_head = 4
n_layer = 3
dropout = 0.1

class Head(nn.Module):
    """One self-attention head with a causal mask."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        weights = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)
        v = self.value(x)
        return weights @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

model = GPTLanguageModel().to(device)
print("Model created.")
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

## 7. Check model output before training

### Explanation
The `logits` tensor contains the model's raw scores for every possible next character at every position. Its shape is:

`(batch size, sequence length, vocabulary size)`

The loss compares these scores with the correct next-character IDs. Before training, the loss is usually relatively high because the model starts with randomly initialized parameters.

In [ ]:
sample_x, sample_y = get_batch("train")
logits, loss = model(sample_x, sample_y)
print("Logits shape (batch, sequence, vocabulary):", logits.shape)
print("Initial loss:", float(loss))
print("Expected shape:", (batch_size, block_size, vocab_size))

## 8. Train the model

Training loss should generally decrease. Because the dataset is very small, the model may memorize it; that is expected for this demonstration.

### Explanation
Training repeats these steps:
1. Get a batch of input sequences and their next-character targets.
2. Run the model to calculate prediction scores (`logits`).
3. Calculate cross-entropy loss. Lower loss means the predictions are closer to the correct targets on average.
4. Clear old gradients.
5. Use `backward()` to calculate gradients.
6. Use the AdamW optimizer to update the model parameters.

The validation loss is measured on held-out text. If training loss falls while validation loss rises, the model may be memorizing the small training set (overfitting).

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}
    for split in ["train", "val"]:
        losses = torch.zeros(20)
        for k in range(20):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        results[split] = losses.mean().item()
    model.train()
    return results

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = []

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        history.append({"step": step, **losses})
        print(f"step {step:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training complete.")

## 9. Plot training progress

### Explanation
The graph shows training loss and validation loss over training steps. A downward trend means the model is learning patterns from the training text. The two lines do not need to be identical. With this tiny dataset, the graph is only a learning demonstration, not proof of real-world language ability.

In [ ]:
import matplotlib.pyplot as plt

if history:
    steps = [item["step"] for item in history]
    train_losses = [item["train"] for item in history]
    val_losses = [item["val"] for item in history]
    plt.figure(figsize=(8, 4))
    plt.plot(steps, train_losses, marker="o", label="Train loss")
    plt.plot(steps, val_losses, marker="o", label="Validation loss")
    plt.xlabel("Training step")
    plt.ylabel("Loss")
    plt.title("GPT Training Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

## 10. Generate text from a prompt

The prompt must use characters from the training vocabulary.

### Explanation
`predict_next_token()` takes the prompt, converts it into character IDs, and asks the model for scores at the final position. It selects the character with the highest score and reports its probability.

`generate()` repeatedly:
1. Looks at the most recent context.
2. Calculates next-character probabilities.
3. Samples one next character.
4. Adds it to the context.
5. Repeats to create a longer sequence.

Because this is a character-level model trained on a tiny corpus, generated text may be repetitive or incorrect. That is normal for this starter project.

In [ ]:
@torch.no_grad()
def predict_next_token(prompt):
    model.eval()
    prompt_ids = encode(prompt)
    if not prompt_ids:
        raise ValueError("Prompt has no characters from the training vocabulary.")
    context = torch.tensor([prompt_ids[-block_size:]], dtype=torch.long, device=device)
    logits, _ = model(context)
    next_logits = logits[0, -1, :]
    probs = F.softmax(next_logits, dim=-1)
    next_id = int(torch.argmax(probs).item())
    model.train()
    return itos[next_id], float(probs[next_id].item())

prompt = "Artificial intelligence"
next_char, confidence = predict_next_token(prompt)
print("Prompt:", prompt)
print("Predicted next character:", repr(next_char))
print("Probability of selected character:", round(confidence, 4))

start = "Artificial"
context = torch.tensor([encode(start)], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=200)
print("\nGenerated text:")
print(decode(generated[0].tolist()))

## 11. Save the trained model and tokenizer

This saves a checkpoint and tokenizer vocabulary. Keep these files for the later API deployment step.

### Explanation
The checkpoint stores the learned model weights, the model configuration and the character-to-ID mappings. The tokenizer file stores the mappings separately, and the text file stores the training corpus used in this demonstration.

The model architecture code is also needed later to load the checkpoint in an API. A weights file alone is not enough to run predictions.

In [ ]:
checkpoint_path = OUT_DIR / "gpt_decoder_model.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "model_config": {
        "vocab_size": vocab_size,
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
    },
    "stoi": stoi,
    "itos": itos,
}, checkpoint_path)

with open(OUT_DIR / "tokenizer.json", "w", encoding="utf-8") as f:
    json.dump({"stoi": stoi, "itos": {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "training_text.txt", "w", encoding="utf-8") as f:
    f.write(text_data)

print("Saved files:")
for p in OUT_DIR.iterdir():
    print(p.name, "-", p.stat().st_size, "bytes")

## 12. Download the model files

Run this cell to download the checkpoint to your computer. You can later upload the files to your model hosting service. Do not upload private data or secrets to GitHub.

### Explanation
This cell downloads the saved files from the Colab runtime to your computer. Keep them in a project folder. Later, we will prepare a separate inference API and connect a Vercel website to that API. Do not publish private data, passwords, or API keys in GitHub.

In [ ]:
from google.colab import files
files.download(str(checkpoint_path))
files.download(str(OUT_DIR / "tokenizer.json"))
files.download(str(OUT_DIR / "training_text.txt"))

## 13. Project understanding — important viva points

1. **GPT:** Generative Pre-trained Transformer.
2. **Decoder-only:** This model uses the Transformer decoder-style stack, with causal self-attention.
3. **Token:** A unit of text. This starter notebook uses one character per token.
4. **Query (Q):** Represents what the current token is looking for.
5. **Key (K):** Represents what information a token offers for matching.
6. **Value (V):** Carries the information used in the attention output.
7. **Causal mask:** Prevents a position from looking at future tokens.
8. **Loss:** Measures how far the model's predictions are from the correct next tokens.
9. **Softmax:** Converts output scores into probabilities.
10. **Next-token prediction:** The model predicts the next character in this educational character-level implementation.

## Next stage: GitHub → model API hosting → Vercel

Do not put the large `.pt` checkpoint in a normal Vercel frontend deployment. In the next stage, we will create a small inference API that loads this checkpoint, host that API separately, and connect the Vercel frontend to it.